# 14.3 - Planning

Status: VERIFIED

## What Are We Solving?
Planning adds a phase where the LLM decomposes a complex task into steps before executing them, rather than acting reactively. This improves accuracy on multi-step tasks.

## Mental Model
User request -> LLM generates plan (list of steps) -> Execute each step -> Synthesize answer

In [1]:
import os
import json
from groq import Groq
from dotenv import load_dotenv

load_dotenv()  # loads from .env in project root
client = Groq(api_key=os.getenv("GROQ_API_KEY"))
MODEL = "qwen/qwen3.8-27b"

# Quick test
r = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Say 'groq ok' only"}],
    max_tokens=10
)
print(f"Groq connected: {r.choices[0].message.content.strip()}")
print(f"Model: {MODEL}")

Groq connected: groq ok
Model: qwen/qwen3.8-27b


## Task Decomposition

In [2]:
def plan_task(task: str) -> list:
    """Have the LLM decompose a task into steps."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": (
                "Given a task, return a numbered list of concrete steps to complete it. "
                "Return ONLY the numbered list, nothing else. Max 5 steps."
            )},
            {"role": "user", "content": task}
        ]
    )
    text = response.choices[0].message.content
    steps = [line.strip() for line in text.split("\n") if line.strip() and line.strip()[0].isdigit()]
    return steps

# Test
task = "Research Python async patterns, write example code, and explain it"
plan = plan_task(task)
print(f"Task: {task}")
print(f"Plan ({len(plan)} steps):")
for i, s in enumerate(plan, 1):
    print(f"  {i}. {s}")

Task: Research Python async patterns, write example code, and explain it
Plan (5 steps):
  1. 1. Identify core Python async patterns such as sequential execution, concurrent tasks using `asyncio.gather`, and `async/await` chains within loop constructs.
  2. 2. Draft concise example code snippets for each pattern, ensuring proper use of the `async def` syntax, `await` keywords, and event loop management.
  3. 3. Structure the explanations to define how event loops handle concurrency, blocking I/O, and task scheduling for each specific code example.
  4. 4. Highlight common pitfalls in the explanations, such as blocking calls in async functions or incorrect handling of unhandled exceptions in concurrent tasks.
  5. 5. Review the code and text for clarity, ensuring that theoretical concepts are directly mapped to the provided syntax examples for easy comprehension.


## Plan-Guided Execution

In [3]:
def execute_step(step: str) -> str:
    """Execute a single plan step."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "Execute this step and return the result. Be concise."},
            {"role": "user", "content": step}
        ]
    )
    return response.choices[0].message.content

def run_with_plan(task: str) -> dict:
    """Plan then execute."""
    # Plan
    steps = plan_task(task)
    print("Plan:")
    for i, s in enumerate(steps, 1):
        print(f"  {i}. {s}")
    
    # Execute
    results = []
    for i, step in enumerate(steps, 1):
        result = execute_step(step)
        results.append({"step": i, "description": step, "result": result[:200]})
        print(f"\n  Step {i} result: {result[:100]}...")
    
    return {"steps": steps, "results": results}

output = run_with_plan("Explain the difference between lists and tuples in Python")
print(f"\nCompleted {len(output['results'])} steps")

Plan:
  1. 1. Define immutability: State that lists are mutable (items can be added, removed, or changed after creation) while tuples are immutable (items cannot be changed after creation).
  2. 2. Compare syntax: Note that lists use square brackets `[]` and tuples use parentheses `()` for initialization.
  3. 3. Discuss performance: Explain that tuples are generally faster and more memory-efficient because their immutability allows Python to optimize storage and hashing.
  4. 4. Highlight use cases: Specify that lists are for data collections that may change, whereas tuples are for fixed data structures like coordinates or database records.
  5. 5. Mention hashability: Clarify that tuples can be used as dictionary keys (if their contents are hashable) because they are immutable, but lists cannot.



  Step 1 result: **1. Define Immutability**
Lists are **mutable** data structures, meaning their items can be added, ...



  Step 2 result: ```python
# Lists use square brackets []
my_list = [1, 2, 3]

# Tuples use parentheses ()
my_tuple =...



  Step 3 result: Tuples outperform lists due to their **immutability**, which grants Python’s runtime several optimiz...



  Step 4 result: **Highlight Use Cases:**

*   **Lists** are ideal for **data collections that may change** (mutable)...



  Step 5 result: ### 5. Mention Hashability

Because tuples are **immutable**, they are **hashable**, which means the...

Completed 5 steps


In [4]:
# Verification
assert len(plan) > 0, "Must produce a plan"
assert len(output["results"]) > 0, "Must execute steps"
print("VERIFICATION PASSED: Phase 14.3 complete")

VERIFICATION PASSED: Phase 14.3 complete
